In [10]:
# DSA 210 - Phase 2
# Turkish funds vs bank deposit rate (2021-2025)
# Ulas Alpandiner - 33831

import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# load the two files
tefas_file = glob.glob("TEFAS*.csv")[0]
evds_file  = glob.glob("EVDS*.xlsx")[0]
print("TEFAS :", tefas_file)
print("EVDS  :", evds_file)

funds = pd.read_csv(tefas_file, parse_dates=["Tarih"])
funds.columns = ["Date", "Code", "Name", "Price", "Shares", "Investors", "TotalValue"]

# deposit rates from CBRT (weekly)
rates = pd.read_excel(evds_file).iloc[:, :2]
rates.columns = ["RawDate", "DepositAPR"]
rates["Date"] = pd.to_datetime(rates["RawDate"], errors="coerce", dayfirst=True)
rates["DepositAPR"] = pd.to_numeric(rates["DepositAPR"], errors="coerce")
rates = rates.dropna(subset=["Date", "DepositAPR"])[["Date", "DepositAPR"]]

# turn weekly rate into daily rate so we can match with fund data
daily_idx = pd.date_range(rates["Date"].min(), funds["Date"].max(), freq="D")
rates = rates.set_index("Date").reindex(daily_idx).ffill().rename_axis("Date").reset_index()
rates["Rf"] = rates["DepositAPR"] / 100 / 365

# put each fund into a category based on its name
def categorise(name):
    n = str(name).upper()
    if "PARA PİYASASI" in n: return "Money Market"
    if "ALTIN" in n or "KIYMETLİ MADEN" in n: return "Gold"
    if "HİSSE" in n: return "Equity"
    if "EUROBOND" in n: return "Eurobond"
    if "BORÇLANMA" in n: return "Debt"
    if "FON SEPETİ" in n: return "Fund of Funds"
    if "KARMA" in n: return "Mixed"
    if "DEĞİŞKEN" in n: return "Variable"
    if "KATILIM" in n: return "Participation"
    if "SERBEST" in n: return "Hedge"
    return "Other"

funds["Category"] = funds["Name"].apply(categorise)

# calculate daily returns for each fund
funds = funds.sort_values(["Code", "Date"])
funds["Return"] = funds.groupby("Code")["Price"].pct_change()
funds = funds[funds["Return"].abs() < 0.5]   # drop weird values (probably data errors)

# excess return = fund return - risk free rate
funds = funds.merge(rates[["Date", "Rf"]], on="Date", how="left")
funds["Excess"] = funds["Return"] - funds["Rf"]

print("Observations:", len(funds), "| Funds:", funds["Code"].nunique())


# ----- EDA -----

# plot 1: how the deposit rate changed over time
plt.figure(figsize=(9, 4))
plt.plot(rates["Date"], rates["DepositAPR"], color="red")
plt.title("CBRT Weighted-Average TL Deposit Rate (APR %)")
plt.grid(alpha=.3); plt.tight_layout()
plt.savefig("fig1_deposit_rate.png"); plt.close()

# plot 2: if i put 1 TL in each category, how much would it grow?
cat_daily = funds.groupby(["Date", "Category"])["Return"].mean().unstack().sort_index()
cat_cum = (1 + cat_daily.fillna(0)).cumprod()
deposit_cum = (1 + rates.set_index("Date")["Rf"].reindex(cat_cum.index).fillna(0)).cumprod()

plt.figure(figsize=(10, 6))
for c in cat_cum.columns:
    plt.plot(cat_cum.index, cat_cum[c], label=c)
plt.plot(deposit_cum.index, deposit_cum.values, "k--", lw=2, label="Deposit")
plt.yscale("log")
plt.title("Cumulative Growth of 1 TL by Category vs Deposit (log)")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig("fig2_cumulative.png"); plt.close()

# table: annualised return, volatility, sharpe ratio for each category
summary = funds.groupby("Category")["Return"].agg(
    AnnReturn=lambda x: x.mean() * 252,
    AnnVol=lambda x: x.std() * np.sqrt(252),
)
rf_ann = rates["Rf"].mean() * 252
summary["Sharpe"] = (summary["AnnReturn"] - rf_ann) / summary["AnnVol"]
summary = summary.sort_values("Sharpe", ascending=False)
print("\nRisk/Return summary:")
print(summary.round(3))
summary.to_csv("summary.csv")

# plot 3: risk vs return scatter plot
plt.figure(figsize=(9, 6))
plt.scatter(summary["AnnVol"], summary["AnnReturn"], s=80)
for cat, row in summary.iterrows():
    plt.annotate(cat, (row["AnnVol"], row["AnnReturn"]), fontsize=9,
                 xytext=(5, 5), textcoords="offset points")
plt.axhline(rf_ann, color="red", ls="--", label=f"Deposit {rf_ann*100:.1f}%")
plt.xlabel("Annualised volatility"); plt.ylabel("Annualised return")
plt.title("Risk / Return by Category"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig3_risk_return.png"); plt.close()


# ----- prepare sample for hypothesis tests -----
# for each fund, take its average excess return (need at least 60 days of data)
# also skip categories with very few funds

per_fund = (funds.groupby(["Code", "Category"])["Excess"]
                 .agg(["mean", "count"]).reset_index())
per_fund = per_fund[per_fund["count"] >= 60]
counts = per_fund["Category"].value_counts()
per_fund = per_fund[per_fund["Category"].isin(counts[counts >= 5].index)]


# ----- TEST 1: one-sample t-test -----
# for each category, test if the average excess return is different from 0
# H0: mean excess = 0 (same as deposit)
# H1: mean excess != 0

print("\n=== TEST 1: One-sample t-test per category ===")
print("H0: mean excess return = 0 (category matches deposit)")
print(f"{'Category':<18}{'n':>5}{'AnnMean':>10}{'t':>8}{'p':>12}  decision")
t_results = []
for cat, grp in per_fund.groupby("Category"):
    x = grp["mean"].values
    t, p = stats.ttest_1samp(x, 0)
    dec = "REJECT H0" if p < 0.05 else "fail to reject"
    print(f"{cat:<18}{len(x):>5}{x.mean()*252:>9.2%}{t:>8.2f}{p:>12.2e}  {dec}")
    t_results.append([cat, len(x), x.mean()*252, t, p, dec])

pd.DataFrame(t_results, columns=["Category","n","AnnMean","t","p","Decision"]
            ).to_csv("test1_ttest.csv", index=False)


# ----- TEST 2: chi-square test of independence -----
# is there a relationship between the category and whether a fund beats the deposit?
# H0: category and beat-the-deposit are independent
# H1: they are dependent

print("\n=== TEST 2: Chi-square test of independence ===")
print("H0: Category and beating-the-deposit are independent")

# label each fund: did it beat the deposit or not?
per_fund["Beat"] = np.where(per_fund["mean"] > 0, "Beat", "NotBeat")
table = pd.crosstab(per_fund["Category"], per_fund["Beat"])

# chi-square needs every expected count to be at least 5
# so drop categories that are too small
while True:
    _, _, _, exp = stats.chi2_contingency(table)
    if exp.min() >= 5:
        break
    worst_row = np.argmin(exp.min(axis=1))
    dropped = table.index[worst_row]
    print(f"  dropping '{dropped}' (too few funds)")
    table = table.drop(dropped)

print("\nContingency table (observed counts):")
print(table)

chi2, p_chi, dof, expected = stats.chi2_contingency(table)

print("\nExpected counts (under H0):")
print(pd.DataFrame(expected, index=table.index, columns=table.columns).round(1))

print(f"\nSmallest expected count: {expected.min():.1f}  (OK, >= 5)")
print(f"\nChi-square statistic : {chi2:.3f}")
print(f"Degrees of freedom   : {dof}  =  (R-1) x (C-1) = "
      f"({table.shape[0]}-1) x ({table.shape[1]}-1)")
print(f"p-value              : {p_chi:.3e}")
print("=> REJECT H0: category and performance are DEPENDENT."
      if p_chi < 0.05 else "=> Fail to reject H0.")

table.to_csv("test2_contingency.csv")




TEFAS : TEFAS_Tum_Veriler_Raw.csv
EVDS  : EVDS_14-04-2026__1.xlsx
Observations: 1344884 | Funds: 2002

Risk/Return summary:
               AnnReturn  AnnVol  Sharpe
Category                                
Money Market       0.364   0.055   1.874
Gold               0.550   0.247   1.165
Eurobond           0.458   0.183   1.072
Other              0.491   0.219   1.046
Fund of Funds      0.438   0.174   1.014
Variable           0.437   0.178   0.984
Debt               0.328   0.072   0.921
Mixed              0.460   0.220   0.902
Hedge              0.432   0.213   0.799
Equity             0.510   0.375   0.661
Participation      0.333   0.119   0.606

=== TEST 1: One-sample t-test per category ===
H0: mean excess return = 0 (category matches deposit)
Category              n   AnnMean       t           p  decision
Debt                 74    5.35%    7.31    2.72e-10  REJECT H0
Equity              395   12.63%    2.85    4.54e-03  REJECT H0
Eurobond              9   20.06%   43.67    8.35e

In [ ]:
from google.colab import drive
drive.mount('/content/drive')